# 📱 Krea 2 & 顶级日漫手绘工作站（手机远程一键懒人版）

**专为手机远程操作优化！所有大模型（Krea 2）与前端界面（ComfyUI）都会在你点击 `▶` 后由谷歌云端机房以千兆光纤自动下载安装，无需占用你自己的电脑硬盘！**

> **💡 唯一需要检查的设置**：点击右上角 `连接 (Connect)` 旁的小箭头 `▼` $\rightarrow$ `更改运行时类型` $\rightarrow$ 选 **`A100 GPU`**（练 LoRA 满血首选）或 **`L4 GPU`**（日常画图首选）。

In [ ]:
#@title ## 🎨 【按钮 1】自动下载 Krea 2 全套模型 + 启动 ComfyUI 可视化前端（点左侧 ▶ 即可）
PRECISION = "FP8 高速版 (推荐, L4/A100通杀, 秒下载)" #@param ["FP8 高速版 (推荐, L4/A100通杀, 秒下载)", "BF16 无损原版 (A100 40G/80G 专属)"]
#@markdown 点一下左边的 `▶`，谷歌云端服务器会自动完成：\
#@markdown 1. 挂载你的 Google Drive 云盘（保存你的 LoRA 与作品）\
#@markdown 2. 自动安装 **ComfyUI 前端界面**（云端版“秋叶整合包”）\
#@markdown 3. 以机房千兆网速自动下载 **Krea-2-Turbo 主模型 + Qwen3-VL 编码器 + Qwen-Image VAE**\
#@markdown 4. 弹出 `🚀 【点击进入 ComfyUI】` 网页链接，点开就能直接画图！

import os, subprocess, threading, re
print("🚀 按钮 1 已成功触发！正在检测云端显卡（往下滑即可看到实时进度）...", flush=True)
print("🔍 当前分配显卡:", subprocess.getoutput("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader"), flush=True)

USE_GOOGLE_DRIVE = False #@param {type:"boolean"}
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    STUDIO_ROOT = "/content/drive/MyDrive/AI_Manga_Studio"
else:
    STUDIO_ROOT = "/content/AI_Manga_Studio"
for sub in ["datasets", "loras", "checkpoints", "output", "configs"]:
    os.makedirs(f"{STUDIO_ROOT}/{sub}", exist_ok=True)

if not os.path.exists("/content/ComfyUI"):
    print("⚙️ 正在云端安装 ComfyUI 可视化前端及插件管理器...")
    !git clone -q https://github.com/Comfy-Org/ComfyUI.git /content/ComfyUI
    !git clone -q https://github.com/ltdrdata/ComfyUI-Manager.git /content/ComfyUI/custom_nodes/ComfyUI-Manager
    !pip install -q -r /content/ComfyUI/requirements.txt

!apt-get -y install -qq aria2 >/dev/null 2>&1

# 下载 Krea 2 官方 ComfyUI 三件套（主模型 + Qwen3-VL + Qwen-Image VAE）
diff_file = "krea2_turbo_bf16.safetensors" if "BF16" in PRECISION else "krea2_turbo_fp8_scaled.safetensors"
te_file = "qwen3vl_4b_bf16.safetensors" if "BF16" in PRECISION else "qwen3vl_4b_fp8_scaled.safetensors"
vae_file = "qwen_image_vae.safetensors"

def fast_dl(url, dest_dir, filename):
    os.makedirs(dest_dir, exist_ok=True)
    target = os.path.join(dest_dir, filename)
    if not os.path.exists(target):
        print(f"⬇️ 正在通过谷歌机房高速通道下载 {filename} ...")
        !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M "{url}" -d "{dest_dir}" -o "{filename}"
    else:
        print(f"✅ 已存在 {filename}，跳过下载")

base_hf = "https://huggingface.co/Comfy-Org/Krea-2/resolve/main"
fast_dl(f"{base_hf}/diffusion_models/{diff_file}", "/content/ComfyUI/models/diffusion_models", diff_file)
fast_dl(f"{base_hf}/text_encoders/{te_file}", "/content/ComfyUI/models/text_encoders", te_file)
fast_dl(f"{base_hf}/vae/{vae_file}", "/content/ComfyUI/models/vae", vae_file)

!rm -rf /content/ComfyUI/models/loras /content/ComfyUI/models/checkpoints /content/ComfyUI/output
!ln -s "{STUDIO_ROOT}/loras" /content/ComfyUI/models/loras
!ln -s "{STUDIO_ROOT}/checkpoints" /content/ComfyUI/models/checkpoints
!ln -s "{STUDIO_ROOT}/output" /content/ComfyUI/output

wf_dir = "/content/ComfyUI/user/default/workflows"
os.makedirs(wf_dir, exist_ok=True)
!wget -q -O "{wf_dir}/Krea2_Turbo_Official_T2I.json" https://raw.githubusercontent.com/Comfy-Org/workflow_templates/main/templates/image_krea2_turbo_t2i.json

!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb >/dev/null 2>&1

def start_tunnel(port):
    p = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{port}"],
        stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True
    )
    for line in p.stderr:
        m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
        if m:
            print(f"\n{'='*65}")
            print(f"🚀 【点击进入 ComfyUI 漫画工作台】: {m.group(0)}")
            print(f"{'='*65}\n")

threading.Thread(target=start_tunnel, args=(8188,), daemon=True).start()
!python /content/ComfyUI/main.py --dont-print-server

In [ ]:
#@title ## 🔥 【按钮 2】一键全自动训练 Krea-2-Raw 专属画风/角色 LoRA（点左侧 ▶ 即可）
#@markdown 当我在你的 Mac 上帮你扒好 Danbooru 原画、打好自然语言标签并同步后，你只需要点一下左边的 `▶`！\
#@markdown 它会自动在云端下载 `krea/Krea-2-Raw` 底模、拉取你的训练集、根据 A100/L4 自动开启最优配置，并将练好的 LoRA 实时存进你的 Google 云端硬盘。

DATASET_NAME = "imigimuru_makeine" #@param {type:"string"}
LORA_NAME = "krea2_imigimuru_makeine_v1" #@param {type:"string"}
TRIGGER_WORD = "by_imigimuru" #@param {type:"string"}
STEPS = 2000 #@param {type:"integer"}
LEARNING_RATE = "1e-4" #@param ["1e-4", "8e-5", "5e-5"]
LORA_RANK = 32 #@param [16, 32, 64]
AUTO_SHUTDOWN = True #@param {type:"boolean"}
#@markdown > 💡 已默认勾选 `AUTO_SHUTDOWN`：训练一结束、模型存入云盘后，会自动调用 `runtime.unassign()` 立即退租关停显卡，绝不白白多扣你 1 秒钟的算力额度！

import os, subprocess, zipfile, yaml
print("🚀 按钮 2 已成功触发！正在初始化训练环境（请往下滑查看实时进度）...", flush=True)

gpu_info = subprocess.getoutput("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader")
print(f"🔍 当前显卡: {gpu_info}", flush=True)

SAVE_TO_GOOGLE_DRIVE = True #@param {type:"boolean"}
if SAVE_TO_GOOGLE_DRIVE:
    print("📂 正在请求挂载 Google 云盘（请在弹窗中点 Continue，不要切出浏览器 App）...", flush=True)
    from google.colab import drive
    drive.mount('/content/drive')
    STUDIO_ROOT = "/content/drive/MyDrive/AI_Manga_Studio"
else:
    STUDIO_ROOT = "/content/AI_Manga_Studio"
for sub in ["datasets", "loras", "checkpoints", "output", "configs"]:
    os.makedirs(f"{STUDIO_ROOT}/{sub}", exist_ok=True)

# 自动从 GitHub 同步最新打包的数据集（免去手动上传云盘！）
if not os.path.exists("/content/krea2-manga-studio"):
    !git clone -q https://github.com/zsdlz971004-boop/krea2-manga-studio.git /content/krea2-manga-studio
else:
    !cd /content/krea2-manga-studio && git pull -q

extract_dir = f"/content/dataset/{DATASET_NAME}"
os.makedirs(extract_dir, exist_ok=True)

repo_zip = f"/content/krea2-manga-studio/datasets/{DATASET_NAME}.zip"
drive_zip = f"{STUDIO_ROOT}/datasets/{DATASET_NAME}.zip"
target_zip = repo_zip if os.path.exists(repo_zip) else (drive_zip if os.path.exists(drive_zip) else None)

if target_zip:
    print(f"📦 自动加载训练集: {target_zip}")
    with zipfile.ZipFile(target_zip, 'r') as zf:
        zf.extractall(extract_dir)
    print(f"✅ 训练集就绪，共 {len(os.listdir(extract_dir))} 个文件！")
else:
    raise FileNotFoundError(f"未找到训练集 {DATASET_NAME}.zip！请先在聊天窗口告诉 Antigravity 你想练哪位画师，我会自动帮你扒图打标并同步过来。")

if not os.path.exists("/content/ai-toolkit"):
    print("⚙️ 正在安装 ostris/ai-toolkit 训练引擎...")
    !git clone -q https://github.com/ostris/ai-toolkit.git /content/ai-toolkit
    !cd /content/ai-toolkit && git submodule update --init --recursive
    !pip install -q -r /content/ai-toolkit/requirements.txt

!apt-get -y install -qq aria2 >/dev/null 2>&1
raw_ckpt = "/content/krea2_raw_bf16.safetensors"
if not os.path.exists(raw_ckpt):
    print("⬇️ 正在通过免登录高速镜像拉取 Krea-2-Raw 满血底模 (krea2_raw_bf16.safetensors)...")
    !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M "https://huggingface.co/Comfy-Org/Krea-2/resolve/main/diffusion_models/krea2_raw_bf16.safetensors" -d "/content" -o "krea2_raw_bf16.safetensors"

is_a100 = "A100" in gpu_info or "80" in gpu_info or "40960" in gpu_info
quantize = not is_a100
low_vram = not is_a100
print("🚀 模式:", "A100 BF16 满血极速训练" if is_a100 else "L4 24GB FP8 防爆显存安全训练")

config = {
    "job": "extension",
    "config": {
        "name": LORA_NAME,
        "process": [{
            "type": "sd_trainer",
            "training_folder": f"{STUDIO_ROOT}/loras",
            "device": "cuda:0",
            "trigger_word": TRIGGER_WORD,
            "network": {"type": "lora", "linear": int(LORA_RANK), "linear_alpha": int(LORA_RANK)},
            "save": {"dtype": "float16", "save_every": 500, "max_step_saves_to_keep": 4},
            "datasets": [{
                "folder_path": extract_dir,
                "caption_ext": "txt",
                "caption_dropout_rate": 0.05,
                "shuffle_tokens": False,
                "cache_latents_to_disk": True,
                "cache_text_embeddings": True,
                "resolution": [1024]
            }],
            "train": {
                "batch_size": 1,
                "steps": int(STEPS),
                "gradient_accumulation_steps": 1,
                "train_unet": True,
                "train_text_encoder": False,
                "gradient_checkpointing": True,
                "noise_scheduler": "flowmatch",
                "optimizer": "adamw8bit",
                "lr": float(LEARNING_RATE),
                "lr_scheduler": "cosine",
                "dtype": "bf16"
            },
            "model": {
                "name_or_path": raw_ckpt,
                "arch": "krea2",
                "quantize": quantize,
                "low_vram": low_vram
            }
        }]
    }
}

cfg_path = f"{STUDIO_ROOT}/configs/{LORA_NAME}.yaml"
with open(cfg_path, "w", encoding="utf-8") as f:
    yaml.dump(config, f, sort_keys=False, allow_unicode=True)

print(f"🔥 开始全自动训练！权重将自动保存在云盘: {STUDIO_ROOT}/loras/")
!python /content/ai-toolkit/run.py "{cfg_path}"

lora_out_dir = f"{STUDIO_ROOT}/loras/{LORA_NAME}"
saved_files = [f for f in os.listdir(lora_out_dir) if f.endswith('.safetensors')] if os.path.exists(lora_out_dir) else []
if saved_files:
    print(f"🎉 训练成功！已生成 LoRA 权重文件: {saved_files}")
    if AUTO_SHUTDOWN:
        print("✅ 模型已安全落盘！正在自动释放并关停 GPU 运行时，停止消耗计算单元...")
        from google.colab import runtime
        runtime.unassign()
else:
    print("⚠️ 未检测到生成的 .safetensors 权重文件，请查看上方日志定位原因（已保留当前会话未自动关机，方便排查）。")